# 🧠 Memlayer vs Mem0: LoCoMo Benchmark Comparison

**LLM-Driven Salience & Retrieval Evaluation**

This notebook compares **Memlayer** and **Mem0** on the **LoCoMo benchmark** using the official 10-conversation dataset.

## Key Differences

- **Memlayer**: LLM decides both salience (what's relevant) and retrieval method
- **Mem0**: Fixed vector-search approach with built-in filtering

## Dataset: locomo10.json

- 10 long conversations (19-32 sessions each)
- 370-690 turns per conversation
- ~200 QA pairs per conversation
- Categories: Single-hop, multi-hop, temporal reasoning, open-domain, adversarial

## Estimated Runtime

- **Quick test**: 10-15 minutes (first 2 conversations)
- **Full benchmark**: 45-60 minutes (all 10 conversations)

## 🛠️ Setup & Installation

In [ ]:
import subprocess
import sys

print('📦 Installing dependencies...\n')

packages = [
    'git+https://github.com/thebnbrkr/memlayer.git',
    'mem0ai',
    'rouge-score',
    'pandas',
    'matplotlib',
    'seaborn',
    'numpy'
]

for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('✅ Installation complete!')

## 🔑 Configure API Keys

In [ ]:
import os
from getpass import getpass

if 'OPENAI_API_KEY' not in os.environ:
    os.environ['OPENAI_API_KEY'] = getpass('🔑 Enter OpenAI API key: ')
    print('✅ OpenAI API key configured!')
else:
    print('✅ OpenAI API key already configured')

## 📥 Load LoCoMo Dataset

In [ ]:
import json
import requests
from typing import List, Dict, Any

def load_locomo_from_github():
    url = 'https://raw.githubusercontent.com/snap-research/locomo/refs/heads/main/data/locomo10.json'
    try:
        print('📥 Downloading locomo10.json from GitHub...\n')
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        print(f'✅ Downloaded {len(data)} conversations\n')
        return data
    except Exception as e:
        print(f'⚠️  Error downloading: {e}')
        return None

def convert_locomo_to_benchmark(locomo_data: List[Dict]) -> List[Dict]:
    converted = []
    for item in locomo_data:
        conv = item.get('conversation', {})
        sample_id = item.get('sample_id', 'unknown')
        
        all_turns = []
        session_count = 0
        session_keys = sorted([k for k in conv.keys() if k.startswith('session_') and '_date_time' not in k])
        
        for session_key in session_keys:
            session_data = conv.get(session_key, [])
            session_num = int(session_key.split('_')[1])
            datetime_key = f'session_{session_num}_date_time'
            session_datetime = conv.get(datetime_key, 'unknown date')
            
            for turn in session_data:
                all_turns.append({
                    'speaker': turn.get('speaker', ''),
                    'text': turn.get('text', ''),
                    'dia_id': turn.get('dia_id', f'{session_num}:?'),
                    'session': session_num,
                    'session_datetime': session_datetime,
                })
            session_count = session_num
        
        qa_pairs = []
        for qa in item.get('qa', []):
            qa_pairs.append({
                'question': qa.get('question', ''),
                'answer': qa.get('answer', ''),
                'evidence': qa.get('evidence', []),
                'category': qa.get('category', 0),
                'category_name': {1: 'Single-hop', 2: 'Multi-hop', 3: 'Temporal', 4: 'Open-domain', 5: 'Adversarial'}.get(qa.get('category', 0), 'Unknown')
            })
        
        converted.append({
            'conversation_id': sample_id,
            'speaker_a': conv.get('speaker_a', 'Speaker A'),
            'speaker_b': conv.get('speaker_b', 'Speaker B'),
            'turns': all_turns,
            'qa_pairs': qa_pairs,
            'num_sessions': session_count,
            'num_turns': len(all_turns),
            'num_qa': len(qa_pairs),
        })
    return converted

print('='*70)
print('📊 LOADING LOCOMO DATASET')
print('='*70 + '\n')

locomo_raw = load_locomo_from_github()

if locomo_raw:
    conversations = convert_locomo_to_benchmark(locomo_raw)
    print(f'✅ Converted {len(conversations)} conversations')
    total_qa = sum(c['num_qa'] for c in conversations)
    total_turns = sum(c['num_turns'] for c in conversations)
    print(f'\n📈 Dataset Stats: {len(conversations)} convos, {total_qa} QA pairs, {total_turns} turns')
else:
    conversations = []
    print('❌ Failed to load dataset')

print('\n' + '='*70)

## 📏 Evaluation Metrics

In [ ]:
from rouge_score import rouge_scorer
import numpy as np
import re

def normalize_answer(answer):
    if answer is None:
        return ''
    if not isinstance(answer, str):
        answer = str(answer)
    answer = re.sub(r'\\b(a|an|the)\\b', ' ', answer.lower())
    answer = re.sub(r'[^\\w\\s]', ' ', answer)
    answer = ' '.join(answer.split())
    return answer

def calculate_f1(prediction, ground_truth) -> float:
    pred_tokens = set(normalize_answer(prediction).split())
    truth_tokens = set(normalize_answer(ground_truth).split())
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return 0.0 if len(truth_tokens) > 0 else 1.0
    common = pred_tokens & truth_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(truth_tokens)
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_rouge(prediction, ground_truth) -> dict:
    pred_str = str(prediction).lower()
    truth_str = str(ground_truth).lower()
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(truth_str, pred_str)
    return {
        'rouge1': scores['rouge1'].fmeasure,
        'rouge2': scores['rouge2'].fmeasure,
        'rougeL': scores['rougeL'].fmeasure
    }

def evaluate_qa(prediction, ground_truth) -> dict:
    f1 = calculate_f1(prediction, ground_truth)
    rouge = calculate_rouge(prediction, ground_truth)
    exact_match = 1.0 if normalize_answer(prediction) == normalize_answer(ground_truth) else 0.0
    return {'f1': f1, 'rouge1': rouge['rouge1'], 'rouge2': rouge['rouge2'], 'rougeL': rouge['rougeL'], 'exact_match': exact_match}

print('✅ Metrics defined: F1, ROUGE-1/2/L, Exact Match')

## 🧪 Test 1: Memlayer (LLM-Driven Salience & Retrieval)

In [ ]:
import time

print('='*70)
print('🧪 TEST 1: MEMLAYER (LLM-DRIVEN)')
print('='*70 + '\n')

try:
    from memlayer import OpenAI as Memlayer
    
    memlayer_client = Memlayer(
        model='gpt-4o-mini',
        user_id='locomo_test',
        storage_path='./memlayer_benchmark',
        operation_mode='online'
    )
    
    memlayer_results = []
    test_conversations = conversations[:2]
    print(f'Testing on {len(test_conversations)} conversations\n')
    
    for conv_idx, conversation in enumerate(test_conversations):
        conv_id = conversation['conversation_id']
        print(f'📍 Conversation {conv_idx + 1}/{len(test_conversations)} ({conv_id})')
        print(f'   Sessions: {conversation['"'num_sessions'"']}, Turns: {conversation['"'num_turns'"']}')
        
        for turn in conversation['"'turns'"'][:50]:
            message = f'{turn['"'speaker'"']}: {turn['"'text'"'][:100]}'
            memlayer_client.chat([{'"'role'"': '"'user'"', '"'content'"': message}])
        
        print(f'   ✅ Stored turns')
        
        for qa_idx, qa in enumerate(conversation['"'qa_pairs'"'][:5]):
            response = memlayer_client.chat([{'"'role'"': '"'user'"', '"'content'"': qa['"'question'"']}])
            metrics = evaluate_qa(response, qa['"'answer'"'])
            memlayer_results.append({'f1': metrics['"'f1'"'], 'category': qa['"'category_name'"']})
            print(f'   Q{qa_idx + 1}: F1={metrics['"'f1'"']:.3f}')
            time.sleep(0.5)
        
    memlayer_avg_f1 = np.mean([r['"'f1'"'] for r in memlayer_results]) if memlayer_results else 0
    print(f'\n📊 MEMLAYER F1: {memlayer_avg_f1:.3f}')

except Exception as e:
    print(f'❌ Error: {e}')
    memlayer_results = []
    memlayer_avg_f1 = 0

## 🧪 Test 2: Mem0 (Vector-Search Baseline)

In [ ]:
print('\n' + '='*70)
print('🧪 TEST 2: MEM0 (VECTOR-SEARCH)')
print('='*70 + '\n')

try:
    from mem0 import Memory
    
    mem0_config = {
        'vector_store': {
            'provider': 'chroma',
            'config': {
                'collection_name': 'mem0_locomo',
                'path': './mem0_storage'
            }
        }
    }
    mem0_client = Memory.from_config(mem0_config)
    
    mem0_results = []
    test_conversations = conversations[:2]
    
    for conv_idx, conversation in enumerate(test_conversations):
        conv_id = conversation['conversation_id']
        user_id = f'user_{conv_id}'
        print(f'📍 Conversation {conv_idx + 1}/{len(test_conversations)} ({conv_id})')
        
        for turn in conversation['turns'][:50]:
            msg = f'{turn['speaker']}: {turn['text'][:100]}'
            mem0_client.add(msg, user_id=user_id)
        
        print(f'   ✅ Stored turns')
        
        for qa_idx, qa in enumerate(conversation['qa_pairs'][:5]):
            results = mem0_client.search(qa['question'], user_id=user_id, limit=3)
            response = str(results)[:200] if results else 'No info'
            metrics = evaluate_qa(response, qa['answer'])
            mem0_results.append({'f1': metrics['f1'], 'category': qa['category_name']})
            print(f'   Q{qa_idx + 1}: F1={metrics['f1']:.3f}')
            time.sleep(0.5)
    
    mem0_avg_f1 = np.mean([r['f1'] for r in mem0_results]) if mem0_results else 0
    print(f'\n📊 MEM0 F1: {mem0_avg_f1:.3f}')

except Exception as e:
    print(f'❌ Error: {e}')
    mem0_results = []
    mem0_avg_f1 = 0

## 🏆 Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

print('\n' + '='*70)
print('🏆 FINAL COMPARISON')
print('='*70 + '\n')

df = pd.DataFrame([
    {'System': 'Memlayer', 'F1': memlayer_avg_f1},
    {'System': 'Mem0', 'F1': mem0_avg_f1}
])

print(df.to_string(index=False))

if mem0_avg_f1 > 0:
    improvement = ((memlayer_avg_f1 - mem0_avg_f1) / mem0_avg_f1) * 100
    print(f'\n📊 Improvement: {improvement:+.1f}%')
    if improvement > 0:
        print(f'✅ Memlayer WINS by {improvement:.1f}%! 🎉')
    elif improvement < 0:
        print(f'⚠️  Mem0 leads by {abs(improvement):.1f}%')
    else:
        print('➖ Tied performance')

plt.figure(figsize=(10, 5))
plt.bar(df['System'], df['F1'], color=['#2E86AB', '#A23B72'], alpha=0.8, edgecolor='black')
plt.ylabel('Average F1 Score', fontsize=12)
plt.title('Memlayer vs Mem0: LoCoMo Benchmark', fontsize=14, fontweight='bold')
plt.ylim(0, max(df['F1']) * 1.3)
for i, v in enumerate(df['F1']):
    plt.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n✅ Benchmark complete!')